In [0]:
%python

from pyspark.sql import functions as F

CAT     = "dev_insurance"
INBOUND = f"/Volumes/{CAT}/landing/inbound"
OPS     = f"/Volumes/{CAT}/ops"

SOURCES = [
    {"name": "ref_product",       "format": "csv"},
    {"name": "ref_broker",        "format": "csv"},
    {"name": "policyholder",      "format": "csv"},
    {"name": "policy",            "format": "csv"},
    {"name": "premium_bordereau", "format": "csv"},
    {"name": "claim_events",      "format": "json"},
]

def ingest_to_bronze(src: dict):
    name, fmt = src["name"], src["format"]
    path = f"{INBOUND}/{name}"
        # Guard: Auto Loader can't infer a schema from an empty folder or 0-byte files
    try:
        has_data = any(f.size > 0 for f in dbutils.fs.ls(path))
    except Exception:
        has_data = False          # folder doesn't exist yet
    if not has_data:
        print(f"{name}: no data yet - skipped")
        return


    reader = (spark.readStream.format("cloudFiles")
              .option("cloudFiles.format", fmt)
              .option("cloudFiles.schemaLocation", f"{OPS}/schemas/{name}")
              .option("cloudFiles.schemaEvolutionMode", "addNewColumns"))
    if fmt == "csv":
        reader = reader.option("header", "true")
    # no inferColumnTypes -> every business column arrives as STRING (raw, lossless)

    df = (reader.load(f"{INBOUND}/{name}")
          .select("*",
                  F.col("_metadata.file_path").alias("_source_file"),
                  F.col("_metadata.file_modification_time").alias("_file_mod_ts"),
                  F.current_timestamp().alias("_ingest_ts")))

    query = (df.writeStream
             .option("checkpointLocation", f"{OPS}/checkpoints/bronze_{name}")
             .option("mergeSchema", "true")
             .trigger(availableNow=True)
             .toTable(f"{CAT}.bronze.{name}"))
    query.awaitTermination()
    print(f"{name}: done")

for src in SOURCES:
    ingest_to_bronze(src)

In [0]:
SHOW TABLES IN dev_insurance.bronze;
DESCRIBE TABLE dev_insurance.bronze.premium_bordereau;
SELECT * FROM dev_insurance.bronze.premium_bordereau 

In [0]:
SELECT _source_file, count(*) AS lines, count(payment_method) AS with_payment_method
FROM dev_insurance.bronze.premium_bordereau
GROUP BY _source_file ORDER BY _source_file;